# Pilot: is each task worth running?

**GPU: NVIDIA A100.** Runtime > Change runtime type > A100.

For each model (Qwen2.5-7B-Instruct, Phi-4-mini-instruct) and each task with human labels (PubMedQA, Yelp star rating, GSM8K, SAMSum), this notebook trains a small LoRA adapter on 2,000 training examples and scores the base model and the adapter on 200 test items. The last cell writes `level4/pilot/pilot_summary.csv`, says for each task whether the adapter improves on the base model (the task is kept when the gain is at least 3 points of accuracy or exact match, or 2 points of ROUGE-L, on the first model), and estimates the run time of `colab_run.ipynb` from the measured speed.

**Expected run time on an A100: about 2 to 3 h.** The notebook can be rerun after a disconnection: finished adapters and scores are skipped.

## 0. Setup

In [ ]:
import os, subprocess, sys
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
assert GPU, 'No GPU: Runtime > Change runtime type'
EXPECTED = ('A100',)
ALLOW_OTHER_GPU = False   # set to True to run on another GPU anyway (it is recorded in run_info.json)
if not any(g in GPU for g in EXPECTED) and not ALLOW_OTHER_GPU:
    raise SystemExit(f'This notebook is planned for {EXPECTED} (bfloat16, 40 GB or more), got {GPU}: Runtime > Change runtime type')

REPO = '/content/dynalora'
BRANCH = 'paper'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/alexroehrs/dynalora.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='commit %H  %s'
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, and peft 0.21 then refuses to load any LoRA adapter (it requires torchao > 0.16).
# Nothing in this repository uses torchao, so it is removed.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dynalora_results'   # results root (Google Drive); results are not kept in the repository
L4 = f'{RESULTS}/level4'
os.makedirs(L4, exist_ok=True)
sys.path.insert(0, REPO)
from dynalora.runinfo import record_session
e = record_session(f'{L4}/run_info.json', 'pilot notebook start', seeds={'adapter_training': 'per adapter (s<seed> in its name)', 'train_subsets': 42, 'test_subsets': 0, 'bootstrap': 0})
print(e['gpu'], e.get('gpu_memory_gb'), e['git'], e['libraries'], sep='\n')
import torch
assert torch.cuda.is_bf16_supported(), 'this GPU has no bfloat16 support'
GRAD_CKPT = '--gradient_checkpointing' if torch.cuda.get_device_properties(0).total_memory < 60 * 2 ** 30 else ''   # 40 GB: checkpointing in training
print('gradient checkpointing in training:', bool(GRAD_CKPT))
FAILED = []   # steps that did not finish; reported at the end

CPU tests (random tiny models, about 2 min).

In [ ]:
!python -m pytest -q tests/test_tasks.py

In [ ]:
def sh(cmd, log):
    """Runs a shell command, appends its output to `log`, returns the exit code."""
    print(cmd)
    get_ipython().system(f"bash -c 'set -o pipefail; {cmd} 2>&1 | tee -a {log}'")
    return get_ipython().user_ns.get('_exit_code', 1)

def step(name, cmd, log, required=False):
    """Runs one resumable step (the scripts skip what is already saved). A failure stops the notebook if
    `required`, else it is recorded in FAILED and the notebook goes on."""
    rc = sh(cmd, log)
    if rc != 0:
        msg = f'{name} (exit code {rc}, see {log})'
        if required:
            raise RuntimeError(f'{msg}: rerun the notebook to resume')
        FAILED.append(msg); print(f'WARNING: {msg}; going on, rerun the notebook to resume')
    return rc == 0

def tag(model):
    return model.rstrip('/').split('/')[-1]

def eval_bs(model, task):
    """Batch size of the scoring: generation tasks decode short contexts, multiple-choice tasks read long ones."""
    small = 'mini' in model.lower()
    gen = task in ('gsm8k', 'samsum')
    return (32 if gen else 16) if small else (16 if gen else 8)

MODELS = ['Qwen/Qwen2.5-7B-Instruct', 'microsoft/Phi-4-mini-instruct']
TASKS = ['pubmedqa', 'yelp5', 'gsm8k', 'samsum']
P = f'{L4}/pilot'
os.makedirs(P, exist_ok=True)
N_TRAIN, N_VAL, N_TEST, SEED = 2200, 200, 200, 42    # 2,000 training examples + 200 validation examples

In [ ]:
# The four datasets load from the Hub and have the expected fields (a few seconds each; stops here otherwise).
from dynalora import tasks as T
for task in TASKS:
    tr, te = T.load_examples(task, 'train', n=300), T.load_examples(task, 'test', n=20)
    print(f'{task}: {len(tr)} train, {len(te)} test | input: {te[0]["input"][:70]!r} | answer: {te[0]["answer"][:50]!r}')

## 1. Train the pilot adapters and score them (factor 0 = base model, factor 1 = adapter)

In [ ]:
for model in MODELS:
    for task in TASKS:
        name = f'{tag(model)}__{task}__s{SEED}'
        ok = step(f'train {name}', f'python scripts/train_task_chat.py --model {model} --task {task} --seed {SEED} '
                  f'--n_train {N_TRAIN} --n_val {N_VAL} --adapters_dir {P}/adapters {GRAD_CKPT}', f'{P}/train.log')
        if ok:
            step(f'score {name}', f'python scripts/eval_task.py --model {model} --task {task} --adapter_dir {P}/adapters/{name} '
                 f'--out_dir {P}/scores/{name} --base_cache {P}/scores/_base --forms logit --lambdas 0 1 --n_test {N_TEST} '
                 f'--batch_size {eval_bs(model, task)}', f'{P}/score.log')
print('not finished:', FAILED or 'nothing')

## 2. Decision per task and run-time estimate

In [ ]:
import json, numpy as np, pandas as pd
rng = np.random.default_rng(0)
GEN = ('gsm8k', 'samsum')
rows = []
for model in MODELS:
    for task in TASKS:
        name = f'{tag(model)}__{task}__s{SEED}'
        f = f'{P}/scores/{name}/own__logit.csv'
        if not os.path.exists(f):
            continue
        w = pd.read_csv(f).pivot_table(index='item', columns='factor', values='score', aggfunc='first')
        d = (w[1.0] - w[0.0]).values
        b = d[rng.integers(0, len(d), (2000, len(d)))].mean(1)
        thr = 0.02 if task == 'samsum' else 0.03
        t = pd.read_csv(f'{P}/scores/{name}/timing.csv')
        sec1 = t[(t.what == 'score') & (t.form != 'base')].seconds.sum()        # the adapter pass(es) on N_TEST items
        log = json.load(open(f'{P}/adapters/{name}/train_log.json'))
        rows.append(dict(model=tag(model), task=task, base=100 * w[0.0].mean(), adapter=100 * w[1.0].mean(), diff=100 * d.mean(),
                         diff_lo=100 * np.percentile(b, 2.5), diff_hi=100 * np.percentile(b, 97.5), keep=bool(d.mean() >= thr),
                         val_loss_start=log['val_loss'][0][1], val_loss_end=log['val_loss'][-1][1], train_seconds=log['seconds'],
                         score_seconds_per_item=sec1 / len(d)))
S = pd.DataFrame(rows)
S.to_csv(f'{P}/pilot_summary.csv', index=False)
print(S.round(2).to_string(index=False))
print()
first = S[S.model == tag(MODELS[0])]
for r in first.itertuples():
    print(f'{r.task}: ' + ('KEEP' if r.keep else 'REPLACE (reserve tasks: MedQA for a multiple-choice task, XSum for samsum); '
                             'choose a reserve task before running the full notebook'))

# Run-time estimate of colab_run.ipynb (rough: single-pass equivalents x measured seconds per item x 500 items)
N_TRAIN_FULL, N_TEST_FULL = 8000, 500
est = []
for r in S.itertuples():
    gen = r.task in GEN
    train_h = r.train_seconds * (N_TRAIN_FULL - 200) / (N_TRAIN - N_VAL) / 3600
    per_pass = r.score_seconds_per_item * N_TEST_FULL / 3600          # one pass over the test set (choice: the pair of passes)
    own = per_pass * ((2 * 10 + 9 + 1) if gen else (1 + 9 * 0.6))     # own task, full grid, both forms
    cross_as_target = per_pass * ((2 * 3 + 3 + 1) if gen else (1 + 3 * 0.6))   # this task scored for one other adapter, reduced grid
    est.append(dict(model=r.model, task=r.task, train_h=train_h, own_h=own, cross_h_per_adapter=cross_as_target))
E = pd.DataFrame(est)
print(); print(E.round(2).to_string(index=False)); print()
for model in E.model.unique():
    e = E[E.model == model]
    a = e.train_h.sum() + e.own_h.sum() + 3 * e.cross_h_per_adapter.sum()
    print(f'{model}: stage with one seed and all cross-task scores, about {a:.0f} h (plus benchmarks, about 0.5 h per adapter); '
          f'each further seed with own-task scores only, about {e.train_h.sum() + e.own_h.sum():.0f} h')